# Visualization
## Vẽ các biểu đồ phân bố và xu hướng từ dữ liệu Yellow Taxi 2021 đã làm sạch

In [ ]:
from pathlib import Path
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

sns.set(style='darkgrid', palette='muted')
ROOT = Path.cwd()
while ROOT != ROOT.parent and not (ROOT / 'Data').exists():
    ROOT = ROOT.parent
CLEAN_DIR = ROOT / 'Data' / 'processed' / 'clean_data'
FIGURE_DIR = ROOT / 'Data' / 'figures'
FIGURE_DIR.mkdir(parents=True, exist_ok=True)

In [ ]:
files = sorted(CLEAN_DIR.glob('clean_2021-*.parquet'))
df_parts = [pd.read_parquet(path, columns=['tpep_pickup_datetime', 'trip_distance']) for path in files]
df = pd.concat(df_parts, ignore_index=True)
df['tpep_pickup_datetime'] = pd.to_datetime(df['tpep_pickup_datetime'])
df['pickup_hour'] = df['tpep_pickup_datetime'].dt.hour
df.head()

### Phân bố quãng đường
Sử dụng biểu đồ histogram để hiển thị số chuyến đi theo quãng đường.

In [ ]:
plt.figure(figsize=(12, 6))
ax = sns.histplot(df['trip_distance'].clip(upper=40), bins=70, color='#264653')
ax.set(title='Phân bố quãng đường taxi', xlabel='Quãng đường (km)', ylabel='Số chuyến'),
ax.set_ylim(0, 1_200_000)
plt.tight_layout()
plt.savefig(FIGURE_DIR / 'visualization_trip_distance_distribution.png', dpi=200)
plt.show()

### Xu hướng lượng chuyến đi theo giờ
Biểu đồ đường cho thấy chu kỳ theo thời gian và mức độ biến động.

In [ ]:
hourly = df.groupby('pickup_hour').size().reindex(range(24), fill_value=0).reset_index(name='count')
plt.figure(figsize=(14, 6))
ax = sns.lineplot(data=hourly, x='pickup_hour', y='count', marker='o', color='#e9c46a')
ax.set(title='Số chuyến đi theo giờ trong ngày - Yellow Taxi 2021', xlabel='Giờ trong ngày (0-23)', ylabel='Số chuyến đi')
ax.set_ylim(0, int(hourly['count'].max() * 1.05))
plt.xticks(range(24), range(24))
plt.tight_layout()
plt.savefig(FIGURE_DIR / 'visualization_trips_hourly.png', dpi=200)
plt.show()

## Tổng quan dữ liệu
Hiển thị chỉ số tổng số chuyến đi và quãng đường trung bình.

In [ ]:
print('Tổng số chuyến đi:', len(df))
print('Quãng đường trung bình (km):', df['trip_distance'].mean())
print('Số giờ quan sát:', df['pickup_hour'].nunique())